In [9]:
!pip install -q -U google-genai pydantic pandas
import json
import pandas as pd

from google import genai
from getpass import getpass
from pydantic import BaseModel, Field



In [ ]:
API_KEY = getpass("Enter Gemini API key: ")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.6-flash"

In [24]:
ALLOWED = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [25]:
SYSTEM_TEXT = """
You are a careful healthcare operations note classifier.
You are not a clinician.
Classify only the information provided.
Do not invent reasons or medical conclusions.
"""

def build_prompt(note_text, version="CRAFT"):

    if version == "V1":
        return f"Classify this note: {note_text}"

    return f"""
Context:
CareLoop Clinics is analysing reasons for missed appointments.

Role:
You are a healthcare operations note classifier.

Action:
Choose exactly one root_cause from this list:
{ALLOWED}

If the reason is unclear, choose other.
Do not guess.

Tone:
Neutral, concise and factual.

Format:
Return ONLY valid JSON with these four fields:
1. root_cause: string
2. confidence: number between 0 and 1
3. summary: maximum 25 words
4. recommended_action: practical operational suggestion

Note:
{note_text}
"""

In [26]:
class NoteAnalysis(BaseModel):
    root_cause: str
    confidence: float = Field(ge=0, le=1)
    summary: str
    recommended_action: str

In [27]:
def analyse_note(note_text, version="CRAFT"):

    prompt = build_prompt(note_text, version)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=SYSTEM_TEXT + "\n\n" + prompt
    )

    raw_text = response.text.strip()

    # Remove Markdown code fences if Gemini adds them
    if raw_text.startswith("```"):
        raw_text = raw_text.split("\n", 1)[1]
        raw_text = raw_text.rsplit("```", 1)[0].strip()

    # Convert JSON string into Python dictionary
    data = json.loads(raw_text)

    # Validate output using Pydantic
    result = NoteAnalysis(**data)

    # Check allowed root-cause category
    if result.root_cause not in ALLOWED:
        raise ValueError("Invalid root_cause category")

    # Check summary length
    if len(result.summary.split()) > 25:
        raise ValueError("Summary exceeds 25 words")

    return result.model_dump(), raw_text

In [28]:
test_note = """
Patient could not attend because the clinic was too far away
and public transport was unavailable.
"""

data, raw = analyse_note(test_note)

print("Structured result:")
print(data)

print("\nRaw Gemini response:")
print(raw)

Structured result:
{'root_cause': 'distance_or_transport', 'confidence': 1.0, 'summary': 'Patient was unable to attend due to long clinic distance and unavailable public transportation.', 'recommended_action': 'Explore telehealth options or provide transportation support resources for remote patients.'}

Raw Gemini response:
{
  "root_cause": "distance_or_transport",
  "confidence": 1.0,
  "summary": "Patient was unable to attend due to long clinic distance and unavailable public transportation.",
  "recommended_action": "Explore telehealth options or provide transportation support resources for remote patients."
}


In [29]:
notes = pd.read_csv(
    "/content/followup_notes.csv"
).head(10)

print(notes.head())

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


In [30]:
results = []

for _, row in notes.iterrows():

    try:
        data, raw = analyse_note(str(row["note_text"]))

        results.append({
            "note_id": row["note_id"],
            "note_text": row["note_text"],
            **data
        })

        print("Completed:", row["note_id"])

    except Exception as e:
        print("Failed:", row["note_id"], str(e))

Completed: N001
Completed: N002
Completed: N003
Completed: N004
Completed: N005
Completed: N006
Failed: N007 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 12.762295941s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'descrip

In [31]:
import time

time.sleep(15)

row = notes.loc[notes["note_id"] == "N007"].iloc[0]

data, raw = analyse_note(str(row["note_text"]))

results.append({
    "note_id": row["note_id"],
    "note_text": row["note_text"],
    **data
})

print("N007 completed successfully!")

N007 completed successfully!


In [32]:
notes_analysed = pd.DataFrame(results)

notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print("Successfully analysed:", len(notes_analysed))
print(notes_analysed.head())

Successfully analysed: 10
  note_id                                          note_text  \
0    N001  Patient said she never got any reminder messag...   
1    N002  Lives far from the clinic and could not find a...   
2    N003  Patient felt the consultation fee was too high...   
3    N004                                                NaN   
4    N005  Had high fever and body ache on the day, staye...   

              root_cause  confidence  \
0  forgot_or_no_reminder         1.0   
1  distance_or_transport         1.0   
2            cost_or_fee         1.0   
3                  other         1.0   
4   illness_or_emergency         1.0   

                                             summary  \
0  Patient missed the appointment because she did...   
1  Patient missed the appointment due to living f...   
2  Patient skipped the appointment due to high co...   
3  No operational note or reason for the missed a...   
4  Patient missed the appointment due to high fev...   

           